# TSARA on the campaign archive — the rolling state on real data

**Phase 5, the companion to notebook 05.** Notebook 05 rolls data
manufactured inside it, where the true background under every plume is known.
This notebook rolls the real campaign archive, where there is no answer key, so
you can see the same operations on the records they were built for: the methane
baselines of one drive at three windows, the membership rule measured on the
drive's own readings, the canister's windows and the baseline it adopts from
the PTR, with the offset that comes with it, and what a noise scale estimated
from the record follows on a drive: the measurement that moved the noise scale
to Phase 6.

Without an answer key, the evidence is of three kinds:

* **✔ checks against a loop written from the definition**, as in notebook 05.
  They hold on *any* drive day and at *any* parameters, so change them freely;
* **the numbers `docs/METHODS.md` §6 and §2.5 quote**, re-measured here from the
  files and collected in a **ledger** (section 5). A number is only *compared*
  when your parameters match the ones METHODS measured under; otherwise the
  ledger shows it and says so;
* **physical plausibility**, in the figures.

---

## How to use this notebook

1. **Point it at the archive.** Set `TSARA_ARCHIVE` to the directory holding the
   archive's `2024/` and `2026/` trees, then start Jupyter from that shell:

   ```bash
   export TSARA_ARCHIVE=/path/to/Data
   ```

2. **Run everything once** (*Run All*, about a minute; section 3's ten-drive
   load is the slow part).
3. **Two loading cells do the slow part.** *Load one drive* reads a single drive
   day (sections 1, 2 and 4 use it); *Load ten drives* reads all ten 2024 drive
   days (section 3 uses it). Their parameters, such as `DRIVE_DAY`, need a re-run
   of that loading cell and then of the sections below it.
4. **Every section has its own parameters cell.** Change a value and re-run from
   that cell to the next heading; nothing else needs re-running.
5. **"Try it"** notes suggest changes and say what should happen on the archive.

It reads only `2024/NOAA_MobileLab_Drives/` (the ten drive days) and writes only
to a temporary directory. **It is committed without outputs**, because its
outputs are the archive's data.

---
## Setup

The toolkit: the archive gate, imports, the figure style shared by notebooks
01–05, the reference implementations the ✔ checks compare TSARA against, and the
ledger that compares re-measured numbers with `METHODS.md`.

In [ ]:
# =============================================================================
# TOOLKIT. Every section needs this cell.
# =============================================================================
import logging
import os
import tempfile
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

from tsara import setup_logging
from tsara.align import bin_streams_onto_cells
from tsara.config.analysis import BaselineConfig
from tsara.config.loader import load_manifest
from tsara.core.support import CellBounds, stream_cells
from tsara.ingest import ingest_campaign
from tsara.rolling import (
    rolling_quantile,
    rolling_state,
    window_cells,
)

# --- the archive, and only the part of it this notebook may read --------------
ARCHIVE_ENV = "TSARA_ARCHIVE"
if not os.environ.get(ARCHIVE_ENV):
    raise RuntimeError(
        f"This notebook reads the campaign archive, and {ARCHIVE_ENV} is not set. Set it to "
        "the directory holding the archive's 2024/ and 2026/ trees, e.g. "
        f"`export {ARCHIVE_ENV}=/path/to/Data`, start Jupyter from that shell, and run again. "
        "Notebook 05 runs the same operations on generated data and needs nothing."
    )
ARCHIVE = Path(os.environ[ARCHIVE_ENV])
DRIVES = ARCHIVE / "2024" / "NOAA_MobileLab_Drives"
if not DRIVES.is_dir():
    raise RuntimeError(
        f"{ARCHIVE_ENV} is set, but it holds no 2024/NOAA_MobileLab_Drives. "
        "Point it at the directory containing 2024/ and 2026/."
    )

_work = tempfile.TemporaryDirectory()
WORK = Path(_work.name)
SECOND = 1_000_000_000  # TSARA keeps every time as integer nanoseconds

# --- figure style (identical to notebooks 01-05) ----------------------------
C1, C2, C3 = "#0072b2", "#e69f00", "#009e73"  # Okabe-Ito blue, amber, green: colour-blind safe
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "normal",
        "axes.titlelocation": "left",
        "axes.labelsize": 9.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "grid.color": GRID,
        "grid.linewidth": 0.7,
        "grid.linestyle": "-",
        "legend.frameon": False,
        "legend.fontsize": 9,
        "font.size": 9.5,
        "lines.linewidth": 1.4,
        "figure.dpi": 110,
    }
)


def finish(
    ax, title=None, sub=None, ylab=None, xlab=None, legend=False, grid_axis="y", loc="upper right"
):
    """Apply the shared chrome: left-aligned title, muted subtitle, hairline grid."""
    if title:
        ax.set_title(title, pad=19 if sub else 8)
    if sub:
        ax.text(0, 1.025, sub, transform=ax.transAxes, fontsize=8.8, color=MUTED, va="bottom")
    if ylab:
        ax.set_ylabel(ylab)
    if xlab:
        ax.set_xlabel(xlab)
    if grid_axis:
        ax.grid(True, axis=grid_axis, alpha=0.9)
    ax.set_axisbelow(True)
    if legend:
        ax.legend(loc=loc)


def hhmm(ax, fmt="%H:%M"):
    """Label the x axis as clock time; the date belongs in the title."""
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))


def break_gaps(clock, values, tolerance=5.0):
    """Insert a NaN wherever the clock jumps, so matplotlib draws no line across a gap."""
    clock, values = np.asarray(clock), np.asarray(values, dtype=float)
    steps = np.diff(clock).astype("timedelta64[ms]").astype(float)
    gaps = np.where(steps > tolerance * np.median(steps))[0]
    if gaps.size == 0:
        return clock, values
    return np.insert(clock, gaps + 1, clock[gaps]), np.insert(values, gaps + 1, np.nan)


def overview_strip(ax, clock, values, window, label):
    """Draw the whole record thinly and shade the zoomed window: where the figure below sits."""
    clock, values = break_gaps(clock, values)
    ax.plot(clock, values, color=MUTED, lw=0.5)
    ax.axvspan(window[0], window[1], color=C2, alpha=0.25, lw=0)
    ax.set_yticks([])
    ax.set_ylabel(label, fontsize=8, color=MUTED)
    ax.tick_params(axis="x", labelsize=7.5)
    hhmm(ax)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)


def cells_of(stream):
    """Return a stream's cells as CellBounds: integer-nanosecond starts and stops."""
    edges = stream["time_bnds"].values.astype("datetime64[ns]").astype("int64")
    return CellBounds(start_ns=edges[:, 0].copy(), stop_ns=edges[:, 1].copy())


def finite_only(stream, variable):
    """Return the rows of a stream where `variable` holds a value, cells kept.

    The 2024 merge files put every instrument on 1 s rows with a blank
    wherever it had nothing to report; the analyzer's own readings are the
    rows that hold one. `isel` keeps the (time, nv) cells, where a list
    selection would drop them.
    """
    keep = np.flatnonzero(np.isfinite(stream[variable].values))
    return stream.isel(time=keep)


def manifest(name, body):
    """Write a manifest into the scratch directory and load it (validated by Pydantic)."""
    path = WORK / f"{name}.yaml"
    path.write_text(body)
    return load_manifest(path)


def sweep(windows=("2min", "10min", "60min"), quantiles=(0.01, 0.05, 0.10), **more):
    """Return a BaselineConfig for a sweep, the shape `analysis_example.yaml` uses by default."""
    return BaselineConfig.model_validate(
        {"windows": list(windows), "quantiles": list(quantiles), **more}
    )


# --- reference implementations (share no code with TSARA) --------------------
def reference_weighted_quantile(values, weights, q):
    """Return the weighted quantile of one window, from the definition (METHODS §6.3)."""
    values, weights = np.asarray(values, dtype=float), np.asarray(weights, dtype=float)
    use = np.isfinite(values) & (weights > 0)
    if not use.any():
        return np.nan
    order = np.argsort(values[use], kind="stable")
    v, w = values[use][order], weights[use][order]
    cumulative = np.cumsum(w)
    positions = (cumulative - w / 2) / cumulative[-1]
    return float(np.interp(q, positions, v))


def reference_rolling(reading_cells, values, window_s, q, min_readings=1, membership="overlap"):
    """Return the baseline at every reading, one window at a time (METHODS §6.3, §6.4).

    `membership="overlap"` weights each reading by the share of its cell inside
    the window (TSARA's rule); `membership="midpoint"` counts a reading wholly
    if its midpoint lies in the window and not at all otherwise (the rejected
    rule, what pandas' time-based rolling gives).
    """
    values = np.asarray(values, dtype=float)
    n = len(reading_cells)
    width = int(round(window_s * SECOND))
    midpoints = reading_cells.start_ns + (reading_cells.stop_ns - reading_cells.start_ns) // 2
    out, count = np.full(n, np.nan), np.zeros(n, dtype=int)
    for k in range(n):
        start = int(midpoints[k]) - width // 2
        stop = start + width
        if membership == "overlap":
            overlap = np.minimum(reading_cells.stop_ns, stop) - np.maximum(
                reading_cells.start_ns, start
            )
            weight = np.clip(overlap, 0, None).astype(float)
        else:
            weight = ((midpoints >= start) & (midpoints < stop)).astype(float)
        use = (weight > 0) & np.isfinite(values)
        count[k] = int(use.sum())
        if count[k] >= min_readings and use.any():
            out[k] = reference_weighted_quantile(values[use], weight[use], q)
    return out, count


# --- the ledger and the checks -----------------------------------------------
LEDGER = {}  # claim -> (section, documented, measured, status)
CHECKS = {}  # claim -> held?


def documented(section, claim, value, measured, fmt="{}", applies=True, conditions=""):
    """Put a number METHODS quotes beside the same number re-measured just now.

    Compared as printed, at the precision METHODS prints it. `applies` says
    whether the parameters match the ones METHODS measured under; when they do
    not, the number is shown and not compared.
    """
    shown_doc, shown_now = fmt.format(value), fmt.format(measured)
    status = ("agrees" if shown_doc == shown_now else "DIFFERS") if applies else "not compared"
    LEDGER[claim] = (section, shown_doc, shown_now, status)
    note = "" if applies else f"; METHODS measured {conditions}"
    print(f"  [{status:>12}] {claim}: {shown_now}  (METHODS {section}: {shown_doc}{note})")


def check(claim, holds, detail=""):
    """Record one claim about TSARA and print whether it held just now."""
    CHECKS[claim] = bool(holds)
    mark = "✔" if holds else "✘ DOES NOT HOLD:"
    print(f"  {mark} {claim}" + (f"   [{detail}]" if detail else ""))


def same(a, b):
    """Exact equality, NaN matching NaN."""
    return bool(np.array_equal(np.asarray(a), np.asarray(b), equal_nan=True))


def agree(a, b, ulps=4):
    """Equal to the last bit or so: the same blanks, every value within `ulps` last places.

    For TSARA against a reference written independently in this notebook, whose
    np.interp multiplies and divides in a different order and can round the last
    binary digit differently. Bitwise equality is for TSARA against itself.
    """
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if a.shape != b.shape or not np.array_equal(np.isnan(a), np.isnan(b)):
        return False
    finite = np.isfinite(b)
    return bool(np.all(np.abs(a[finite] - b[finite]) <= ulps * np.spacing(np.abs(b[finite]))))


# --- the noise estimate Phase 5 withdrew, re-measured (METHODS §2.5) ----------
def diff_mad_estimate(stream, variable, window, factor=1.5, min_samples=10):
    """Return the robust first-difference noise estimate at every reading, and the floor's share.

    The arithmetic of the estimator the rolling state carried until 2026-09-29,
    when it moved to Phase 6 (the code is at commit 59bd6ef). Kept here only to
    re-measure what METHODS §2.5 and §6.5 quote about it. A sample is the
    absolute difference of two consecutive finite readings, on the cell between
    their midpoints, dropped when they are farther apart than `factor` times the
    median spacing (a dropout); the window centred on each reading takes the
    weighted median of the samples touching it, scaled by 1.4826 / sqrt(2), and
    is blank with fewer than `min_samples`; every estimate is floored at
    delta / sqrt(12), delta being the smallest positive gap between the
    record's values.
    """
    cells = cells_of(stream)
    values = np.asarray(stream[variable].values, dtype=float)
    finite = np.flatnonzero(np.isfinite(values))
    middle = cells.start_ns + (cells.stop_ns - cells.start_ns) // 2
    sample_middle = middle[finite]
    spacing = np.diff(sample_middle)
    keep = spacing <= factor * np.median(spacing)
    samples = np.abs(np.diff(values[finite]))[keep]
    sample_cells = CellBounds(start_ns=sample_middle[:-1][keep], stop_ns=sample_middle[1:][keep])
    rolled = rolling_quantile(
        sample_cells, samples, window_cells(cells, pd.Timedelta(window).value), [0.5]
    )
    sigma = np.where(
        rolled.n_readings >= min_samples, 1.4826 * rolled.values[:, 0] / np.sqrt(2.0), np.nan
    )
    steps = np.diff(np.unique(values[finite]))
    floor = steps[steps > 0].min() / np.sqrt(12.0) if (steps > 0).any() else 0.0
    raised = float(np.mean(sigma[np.isfinite(sigma)] < floor)) if np.isfinite(sigma).any() else 0.0
    return np.where(np.isfinite(sigma), np.maximum(sigma, floor), np.nan), raised


# --- logging, with the archive and scratch paths hidden ----------------------
def hide_paths(record):
    """Show the archive and scratch directories as <archive> and <work> in TSARA's log messages."""
    for real, shown in ((str(WORK), "<work>"), (str(ARCHIVE), "<archive>")):
        if isinstance(record.msg, str):
            record.msg = record.msg.replace(real, shown)
        if isinstance(record.args, tuple):
            record.args = tuple(
                str(a).replace(real, shown) if real in str(a) else a for a in record.args
            )
    return True


for handler in setup_logging(logging.WARNING).handlers:
    handler.addFilter(hide_paths)

DRIVE_DAYS = sorted(p.name for p in DRIVES.iterdir() if p.is_dir())
print("archive found; drive days available:", ", ".join(DRIVE_DAYS))

---
## Load one drive

One drive day of the NOAA mobile laboratory, read the way a user reads the
archive: a manifest, validated, handed to `ingest_campaign`. Every file on a
drive day shares one 1 s merge grid, with a blank wherever an instrument had
nothing to report, so the Picarro's `CH4_ppb` column holds a value in fewer than
half its rows (notebook 04b, §1), and the analyzer's own readings are the rows
that hold one. Two declarations matter here: the iWAS canister file names each
fill's stop time, so its cells are the exact fill intervals; and the NOy-LIF
accuracy column is declared **random** below, so NOy's enhancements carry a
reported sigma where the Picarro's and the PTR's, declaring nothing, say
`unknown` (04b declares it systematic, the file header's reading; either is a
declaration).

In [ ]:
# ---- PARAMETERS (loading one drive) -----------------------------------------
DRIVE_DAY = "20240718"  # any day printed by the toolkit cell; METHODS measured 20240718

In [ ]:
AT_DOCUMENTED_DAY = DRIVE_DAY == "20240718"  # whether ledger numbers for this drive can be compared
drive = ingest_campaign(
    manifest(
        "one_drive",
        f"""
name: drive_{DRIVE_DAY}
base_path: {DRIVES / DRIVE_DAY}
platform: {{kind: mobile, gps_instrument: metnav}}
instruments:
  picarro:
    loader: {{format: icartt, path_template: "USOS-Picarro-CO2-CH4-CO-H2O_MobileLab_*.ict"}}
    variables:
      ch4: {{column: CH4_ppb, role: gas, units: ppb}}
      co2: {{column: CO2_ppm, role: gas, units: ppm}}
  lif:
    loader: {{format: icartt, path_template: "USOS-NOy-LIF_MobileLab_*.ict"}}
    variables:
      noy:
        column: NOy_LIF
        role: gas
        units: ppb
        convert: {{from_unit: pptv, to_unit: ppb, scale: 0.001}}
        uncertainty:
          random: {{mode: reported, column: NOy_LIF_1SigmaAccuracy}}
  ptr:
    loader: {{format: icartt, path_template: "USOS-PTR_MobileLab_*.ict"}}
    variables:
      benzene_ptr: {{column: Benzene_NOAAPTR_ppbv, role: gas, units: ppb, field: benzene}}
  metnav:
    loader: {{format: icartt, path_template: "USOS-MetNav_MobileLab_*.ict"}}
    variables:
      latitude: {{column: GPS_Lat_deg, role: gps_lat, units: degrees_north}}
      longitude: {{column: GPS_Lon_deg, role: gps_lon, units: degrees_east}}
  iwas:
    loader:
      format: icartt
      path_template: "USOS-iWAS_MobileLab_*.ict"
      support: {{stop_column: iWAS_Stop_UTC, method: mean}}
    variables:
      benzene_iwas: {{column: Benzene_ppbv, role: gas, units: ppb, field: benzene}}
      toluene_iwas: {{column: Toluene_ppbv, role: gas, units: ppb}}
""",
    )
)
# The analyzer's own readings: the merge rows where it reported.
picarro = finite_only(drive["picarro"], "ch4")
ptr = finite_only(drive["ptr"], "benzene_ptr")
lif = finite_only(drive["lif"], "noy")
iwas = drive["iwas"]
steps_s = np.diff(cells_of(picarro).start_ns) / SECOND
print(
    f"{DRIVE_DAY}: Picarro {picarro.sizes['time']} readings on 1 s cells, one every "
    f"{np.median(steps_s):.2f} s (median; {steps_s.mean():.2f} s mean); "
    f"PTR {ptr.sizes['time']}; NOy {lif.sizes['time']}; iWAS {iwas.sizes['time']} fills"
)
documented(
    "§2.5",
    "Picarro readings' median spacing on 20240718 (s)",
    2.00,
    float(np.median(steps_s)),
    "{:.2f}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§2.5",
    "Picarro readings' mean spacing on 20240718 (s)",
    2.31,
    float(steps_s.mean()),
    "{:.2f}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§6.3",
    "Picarro CH₄ readings on 20240718",
    8448,
    picarro.sizes["time"],
    "{}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)

---
## 1. One drive, three baselines

**The question.** Notebook 05 §2 showed what a window follows on manufactured
air. On a real drive through a city, with a plume every few minutes, what do
the baselines at 2 min, 10 min and 60 min look like, and does the count rule
have anything to say?

The Picarro reports every 2 or 3 s on this drive (2.31 s on average), so a
2 min window holds about fifty of its readings. The 1st percentile needs a hundred (`METHODS.md` §6.4),
which is the warning below; the 5th and 10th are reported. The checks compare
the state's baseline with the definition applied window by window on the
drive's own readings.

In [ ]:
# ---- PARAMETERS (section 1) --------------------------------------------------
WINDOWS = ("2min", "10min", "60min")  # the sweep's windows
QUANTILES = (0.01, 0.05, 0.10)  # the sweep's quantiles
DRAWN_QUANTILE = 0.05  # the quantile drawn
VIEW_MINUTES = 40  # minutes drawn, centred on the drive's largest reading

In [ ]:
settings = sweep(windows=WINDOWS, quantiles=QUANTILES)
captured = []
handler = logging.Handler()
handler.emit = lambda record: captured.append(record.getMessage())
logging.getLogger("tsara.rolling.state").addHandler(handler)
try:
    state = rolling_state(picarro, instrument="picarro", baseline=settings, variables=["ch4"])
finally:
    logging.getLogger("tsara.rolling.state").removeHandler(handler)
print("the warning, verbatim:\n  " + ("\n  ".join(captured) if captured else "(none)"))
counts = state["n_readings_window_ch4"].values
print(
    "\nreadings per window (min / median / max): "
    + ", ".join(
        f"{w}: {counts[:, i].min()} / {int(np.median(counts[:, i]))} / {counts[:, i].max()}"
        for i, w in enumerate(WINDOWS)
    )
)
blank = (
    state["baseline_ch4"]
    .attrs["tsara_baseline_blank_fraction"]
    .reshape(len(WINDOWS), len(QUANTILES))
)
print("blank share per sweep point (window x quantile):")
for i, w in enumerate(WINDOWS):
    print(f"  {w:>6}: " + "  ".join(f"q={q:g} {blank[i, j]:.0%}" for j, q in enumerate(QUANTILES)))

cells = cells_of(picarro)
readings = picarro["ch4"].values
j = QUANTILES.index(DRAWN_QUANTILE)
for i, window in enumerate(WINDOWS):
    want, count = reference_rolling(
        cells,
        readings,
        pd.Timedelta(window).total_seconds(),
        DRAWN_QUANTILE,
        min_readings=settings.min_readings_for(DRAWN_QUANTILE),
    )
    check(
        f"§1 the {window} baseline at q={DRAWN_QUANTILE:g} is the definition, window by window",
        agree(state["baseline_ch4"].values[:, i, j], want) and same(counts[:, i], count),
    )

# Figure only from here. The readings reach thousands of ppb where the baselines
# differ by tens, so the view is drawn twice: at the readings' scale, then at the
# baselines' own scale with the readings running off it.
clock = picarro["time"].values
largest = int(np.nanargmax(readings))
half = pd.Timedelta(minutes=VIEW_MINUTES / 2)
view = (pd.Timestamp(clock[largest]) - half, pd.Timestamp(clock[largest]) + half)
inside = (clock >= np.datetime64(view[0])) & (clock < np.datetime64(view[1]))
fig, (top, ax, axb) = plt.subplots(
    3,
    1,
    figsize=(10, 7.6),
    sharex=False,
    gridspec_kw={"height_ratios": [0.5, 1.6, 2.6], "hspace": 0.45},
)
overview_strip(top, clock, readings, view, "whole drive")
t, x = break_gaps(clock[inside], readings[inside])
ax.plot(t, x, color=MUTED, lw=0.6, label="CH₄ reading")
for i, (window, colour) in enumerate(zip(WINDOWS, (C1, C2, C3))):
    t, b = break_gaps(clock[inside], state["baseline_ch4"].values[inside, i, j])
    ax.plot(t, b, color=colour, lw=1.4)
finish(
    ax,
    f"{DRIVE_DAY}: the {DRAWN_QUANTILE:g} quantile over three windows, around the drive's "
    "largest reading",
    "at the readings' scale the three baselines are one line along the bottom",
    "CH₄ (ppb)",
    None,
)
hhmm(ax)
drawn = state["baseline_ch4"].values[inside, :, j]
low, high = np.nanmin(drawn), np.nanmax(drawn)
margin = 0.25 * (high - low) + 1.0
t, x = break_gaps(clock[inside], readings[inside])
axb.plot(t, x, color=MUTED, lw=0.5, alpha=0.5, label="CH₄ reading (runs off the panel)")
for i, (window, colour) in enumerate(zip(WINDOWS, (C1, C2, C3))):
    t, b = break_gaps(clock[inside], drawn[:, i])
    axb.plot(t, b, color=colour, lw=1.6, label=f"baseline at {window}")
axb.set_ylim(low - margin, high + margin)
finish(
    axb,
    "The same view at the baselines' own scale",
    "the short window follows the air a plume leaves behind; the long one holds",
    "CH₄ (ppb)",
    None,
)
axb.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
hhmm(axb)
plt.show()

**What to read off the output.** The warning is the count rule at work on a
real cadence: fifty-odd readings in two minutes cannot make a 1st percentile,
so that sweep point is blank at every reading, and the record says so rather
than reporting the minimum of the window as a background. The figure is §6.1
on a drive, and it needs its second panel: at the readings' scale the three
baselines are indistinguishable, since they differ by tens of ppb under plumes
of a thousand. At their own scale, after a sharp plume the 2 min baseline lifts
for a minute or two where the air stays enhanced, the 60 min baseline does not,
and the 10 min one sits between. Which is the right background is not a question the package
answers; which window's ratio belongs to which event is Phase 6 and 7's, and
the sweep keeps all three.

**Try it.** `WINDOWS = ("30s", "2min", "10min")`: a 30 s window holds a dozen
readings on this cadence, so only the 10th percentile is reported there and
the warning grows. `DRIVE_DAY = "20240722"` (re-run the loading cell): another
drive, the same shape; the ledger then shows its numbers without comparing them.

---
## 2. Which readings belong to a window: the rule, measured

**The question.** `METHODS.md` §6.3 chose overlap weighting (each reading
counts by the share of its cell inside the window) over membership by midpoint
(a reading is in or out), and measured how much the two rules differ on this
drive. Is that table still true of the files?

The rule stated there: the 5th percentile; a window centred on every finite
reading's midpoint; **both** rules interpolated at cumulative-weight midpoints
(Hazen's positions), so that the table shows membership alone; "differ" any
difference above 10⁻⁹ ppb. The midpoint rule is the reference implementation's
`membership="midpoint"`; the overlap rule is TSARA's engine. This notebook is
why the rule says "both": the first version of the table had compared TSARA
with pandas' own centred rolling quantile, whose window is open at its start
and closed at its end and which interpolates at (*N* − 1)*q*, and most of its
difference was that convention rather than the membership. The cell also
re-measures that comparison, so the ledger holds both tables: the withdrawn one
reproduced, and the one that stands.

In [ ]:
# ---- PARAMETERS (section 2) --------------------------------------------------
Q = 0.05  # the quantile compared; METHODS measured 0.05

In [ ]:
cells = cells_of(picarro)  # this section's own inputs: the drive's Picarro readings
readings = picarro["ch4"].values
documented_rows = {  # METHODS §6.3, re-stated 2026-09-28 under one interpolation convention
    "2min": (0.43, 0.00, 0.27, 12.00),
    "10min": (0.37, 0.00, 0.05, 2.04),
    "60min": (0.11, 0.00, 0.00, 0.38),
}
withdrawn_rows = {  # the first version of the table, measured against pandas' rolling quantile
    "2min": (0.83, 0.09, 0.67, 21.79),
    "10min": (0.62, 0.04, 0.19, 2.18),
    "60min": (0.27, 0.00, 0.05, 0.59),
}
applies = AT_DOCUMENTED_DAY and Q == 0.05
when = "on 20240718 at q = 0.05"


def against(by_overlap, other):
    """Return (windows compared, share differing, median, p95, max) of |overlap - other|."""
    both = np.isfinite(by_overlap) & np.isfinite(other)
    gap = np.abs(by_overlap - other)[both]
    return (
        int(both.sum()),
        float(np.mean(gap > 1e-9)),
        float(np.median(gap)),
        float(np.percentile(gap, 95)),
        float(gap.max()),
    )


def ledger_rows(section, label, window, doc, row):
    """Put one table row's four numbers beside the ones METHODS quotes."""
    documented(
        section,
        f"{label} at {window}: windows that differ",
        doc[0],
        row[1],
        "{:.0%}",
        applies,
        when,
    )
    documented(
        section,
        f"{label} at {window}: median difference (ppb)",
        doc[1],
        row[2],
        "{:.2f}",
        applies,
        when,
    )
    documented(
        section,
        f"{label} at {window}: p95 difference (ppb)",
        doc[2],
        row[3],
        "{:.2f}",
        applies,
        when,
    )
    documented(
        section,
        f"{label} at {window}: largest difference (ppb)",
        doc[3],
        row[4],
        "{:.2f}",
        applies,
        when,
    )


print(f"{'window':>7} {'windows':>8} {'differ':>7} {'median':>7} {'p95':>7} {'max':>7}   (ppb)")
midpoint_series = pd.Series(readings, index=picarro["time"].values)  # time IS the cell midpoint
for window in ("2min", "10min", "60min"):
    seconds = pd.Timedelta(window).total_seconds()
    by_overlap = rolling_quantile(
        cells, readings, window_cells(cells, int(seconds * SECOND)), [Q]
    ).values[:, 0]
    by_midpoint, _ = reference_rolling(cells, readings, seconds, Q, membership="midpoint")
    row = against(by_overlap, by_midpoint)
    print(f"{window:>7} {row[0]:>8} {row[1]:>7.0%} {row[2]:>7.2f} {row[3]:>7.2f} {row[4]:>7.2f}")
    ledger_rows("§6.3", "midpoint vs overlap", window, documented_rows[window], row)
    # The withdrawn table: pandas' centred rolling quantile, window (start, stop], (N - 1) q.
    by_pandas = midpoint_series.rolling(window, center=True).quantile(Q).values
    ledger_rows(
        "§6.3",
        "the withdrawn table (pandas' rolling quantile)",
        window,
        withdrawn_rows[window],
        against(by_overlap, by_pandas),
    )
check(
    "§2 the two membership rules differ in the second order: the median gap is under a ppb",
    all(
        np.median(
            np.abs(
                rolling_quantile(
                    cells,
                    readings,
                    window_cells(cells, int(pd.Timedelta(w).total_seconds() * SECOND)),
                    [Q],
                ).values[:, 0]
                - reference_rolling(
                    cells, readings, pd.Timedelta(w).total_seconds(), Q, membership="midpoint"
                )[0]
            )[np.isfinite(readings)]
        )
        < 1.0
        for w in ("2min", "10min", "60min")
    ),
)

**What to read off the output.** Both rules give the same baseline almost
everywhere: the median difference is zero at every window, and the rules
differ at all in a shrinking share of windows as the window lengthens. The
only readings the two rules weigh differently are the ones at a window's
edges, which overlap counts at half weight and midpoint membership wholly or
not at all; a difference of ppb appears only where such an edge reading is
among the window's lowest few, so that its half-weight moves the quantile's
position between two low readings. Overlap weighting was chosen because the
package says a value describes an interval, not because of the size of the
effect; what would not be defensible is leaving the rule unstated, since every
window's count depends on it.

**Try it.** `Q = 0.5`: the median does not make the difference smaller. At
2 min more windows differ (about half, against 43 % at the 5th percentile) and
the 95th percentile of the difference grows (0.45 ppb against 0.27): the
readings at a window's edges move the median too. On another drive day the
ledger shows the numbers and does not compare them.

---
## Load ten drives

All ten 2024 drive days, for the canister's windows in section 3: the iWAS
fills and the Picarro's rows, read through one manifest whose path template
spans the day directories.

In [ ]:
ten = ingest_campaign(
    manifest(
        "ten_drives",
        f"""
name: ten_drives
base_path: {DRIVES}
platform: {{kind: mobile, gps_instrument: metnav}}
instruments:
  picarro:
    loader: {{format: icartt, path_template: "*/USOS-Picarro-CO2-CH4-CO-H2O_MobileLab_*.ict"}}
    variables:
      ch4: {{column: CH4_ppb, role: gas, units: ppb}}
  metnav:
    loader: {{format: icartt, path_template: "*/USOS-MetNav_MobileLab_*.ict"}}
    variables:
      latitude: {{column: GPS_Lat_deg, role: gps_lat, units: degrees_north}}
      longitude: {{column: GPS_Lon_deg, role: gps_lon, units: degrees_east}}
  iwas:
    loader:
      format: icartt
      path_template: "*/USOS-iWAS_MobileLab_*.ict"
      support: {{stop_column: iWAS_Stop_UTC, method: mean}}
    variables:
      benzene_iwas: {{column: Benzene_ppbv, role: gas, units: ppb, field: benzene}}
""",
    )
)
fills_all = cells_of(ten["iwas"])
print({name: stream.sizes["time"] for name, stream in ten.items()})
documented("§6.5", "canister fills on the ten drive days", 261, len(fills_all))
documented(
    "§6.2",
    "Picarro rows over the ten drive days (the merge files' 1 s rows)",
    200410,
    ten["picarro"].sizes["time"],
)
one_species = (
    ten["picarro"].sizes["time"] * 3 * 3 * 3 * 8 / 1e6
)  # (baseline, delta, sigma) x 3 windows x 3 quantiles, float64
documented(
    "§6.2",
    "one species' rolling state at 3 x 3 on the Picarro's own rows (MB)",
    43,
    one_species,
    "{:.0f}",
)

---
## 3. The canister's windows, and the baseline it adopts

**The question.** `METHODS.md` §6.5 measured how many fills of the iWAS
canister a window centred on each fill holds, over all 261 fills of the ten
drive days, and concluded that a canister species has a rolling-quantile
baseline only at windows of hours. Is the table still true, and what does
`from_field` give the canister at the windows it cannot see?

The counting rule, as stated there: a window centred on each fill; fills whose
midpoints fall inside; the fill itself included. Then, on the one drive loaded
above, the three calls of notebook 05 §5: roll the PTR, join its benzene
baseline onto the canister's fill cells, roll the canister with the product
provided.

In [ ]:
# ---- PARAMETERS (section 3) --------------------------------------------------
WINDOWS = ("2min", "10min", "60min", "6h")  # the windows the fills are counted in
ADOPT_WINDOWS = ("10min", "60min")  # the sweep the canister adopts the PTR's baseline at
QUANTILE = 0.05

In [ ]:
documented_fills = {
    "2min": (1, 1, 4, 0.93, 1.00),
    "10min": (1, 1, 5, 0.54, 1.00),
    "60min": (1, 6, 11, 0.004, 0.30),
    "6h": (8, 23, 35, 0.00, 0.00),
}
mid = fills_all.start_ns + (fills_all.stop_ns - fills_all.start_ns) // 2
print(f"{'window':>7} {'min':>4} {'median':>7} {'max':>4} {'only itself':>12} {'fewer than 5':>13}")
for window in WINDOWS:
    half_ns = int(pd.Timedelta(window).value // 2)
    inside = np.array([int(np.sum((mid >= m - half_ns) & (mid < m + half_ns))) for m in mid])
    row = (
        inside.min(),
        int(np.median(inside)),
        inside.max(),
        float(np.mean(inside == 1)),
        float(np.mean(inside < 5)),
    )
    print(f"{window:>7} {row[0]:>4} {row[1]:>7} {row[2]:>4} {row[3]:>12.1%} {row[4]:>13.0%}")
    doc = documented_fills.get(window)
    if doc:
        documented(
            "§6.5",
            f"fills in a {window} window: min / median / max",
            f"{doc[0]} / {doc[1]} / {doc[2]}",
            f"{row[0]} / {row[1]} / {row[2]}",
        )
        documented(
            "§6.5",
            f"fills in a {window} window: only itself",
            doc[3],
            row[3],
            "{:.1%}" if window == "60min" else "{:.0%}",
        )
        documented("§6.5", f"fills in a {window} window: fewer than 5", doc[4], row[4], "{:.0%}")

# On the one drive: what the canister can do alone, and what it adopts.
adopt = sweep(
    windows=ADOPT_WINDOWS,
    quantiles=(QUANTILE,),
    methods={"iwas.benzene_iwas": {"method": "from_field", "instrument": "ptr"}},
)
own = rolling_state(
    iwas,
    instrument="iwas",
    baseline=sweep(windows=ADOPT_WINDOWS, quantiles=(QUANTILE,)),
    variables=["benzene_iwas"],
)
own_blank = own["baseline_benzene_iwas"].attrs["tsara_baseline_blank_fraction"]
print(
    f"\n{DRIVE_DAY}: the canister's own {QUANTILE:g} quantile is blank at {own_blank.tolist()} "
    f"of its {iwas.sizes['time']} fills, per window "
    f"(it needs {adopt.min_readings_for(QUANTILE)} fills)"
)
donor = rolling_state(ptr, instrument="ptr", baseline=adopt, variables=["benzene_ptr"])
joined = bin_streams_onto_cells(
    {"ptr": donor}, stream_cells(iwas, "iwas"), [("ptr", "baseline_benzene_ptr")]
)
adopter = rolling_state(
    iwas,
    instrument="iwas",
    baseline=adopt,
    variables=["benzene_iwas"],
    provided={"benzene_iwas": joined},
)
base = adopter["baseline_benzene_iwas"]
print(
    f"adopted: {base.attrs['tsara_baseline_method']} from {base.attrs['tsara_baseline_from']}; "
    f"the join's record {base.attrs['tsara_support_transform']}, "
    f"{base.attrs['tsara_n_readings']} PTR readings behind {iwas.sizes['time']} fills"
)
check(
    "§3 the canister's own rolling quantile is blank exactly where its window holds too few fills",
    same(
        np.isnan(own["baseline_benzene_iwas"].values[:, :, 0]),
        own["n_readings_window_benzene_iwas"].values < adopt.min_readings_for(QUANTILE),
    ),
)
check(
    "§3 the adopted baseline is the PTR's, joined onto the fill cells, exactly",
    same(base.values, joined["baseline_benzene_ptr"].values),
)
check(
    "§3 the canister's enhancement is its fill minus the adopted baseline",
    same(
        adopter["enhancement_benzene_iwas"].values,
        iwas["benzene_iwas"].values[:, None, None] - base.values,
    ),
)

# What the adoption hands the canister (METHODS §6.5). A 5th percentile of the PTR's
# readings sits about z_q times the PTR's noise below the background, and the
# canister's enhancements inherit that offset. Measured beside it: whether the two
# instruments agree on the same air, the PTR averaged over each fill against the fill.
ptr_noise, _ = diff_mad_estimate(ptr, "benzene_ptr", "10min")
noise_median = float(np.nanmedian(ptr_noise))
offset = -norm.ppf(QUANTILE) * noise_median
long_baseline = donor["baseline_benzene_ptr"].values[:, -1, 0]
negative = float(np.mean(long_baseline[np.isfinite(long_baseline)] < 0))
lowest_fill = float(np.nanmin(iwas["benzene_iwas"].values))
over_fill = bin_streams_onto_cells(
    {"ptr": ptr}, stream_cells(iwas, "iwas"), [("ptr", "benzene_ptr")]
)["benzene_ptr"].values
fills = iwas["benzene_iwas"].values
both = np.isfinite(over_fill) & np.isfinite(fills)
difference = float(np.median(fills[both] - over_fill[both]))
ratio = float(np.median(fills[both] / over_fill[both]))
slope = float(np.polyfit(over_fill[both], fills[both], 1)[0])
print(
    f"\nthe PTR's benzene noise estimate (diff_mad, 10 min, median) is {noise_median:.3f} ppb, so "
    f"its {QUANTILE:g} quantile sits about {offset:.3f} ppb below the background; its "
    f"{ADOPT_WINDOWS[-1]} baseline is negative at {negative:.0%} of its readings; the "
    f"canister's lowest fill is {lowest_fill:.3f} ppb"
)
print(
    f"same air, {both.sum()} fills: canister minus PTR median {difference:+.3f} ppb, ratio "
    f"{ratio:.2f}, least-squares slope of canister on PTR {slope:.2f}"
)
at_rule = AT_DOCUMENTED_DAY and tuple(ADOPT_WINDOWS) == ("10min", "60min") and QUANTILE == 0.05
when = "on 20240718, 10 and 60 min, q = 0.05"
documented(
    "§6.5",
    "PTR benzene noise estimate, median (ppb)",
    0.032,
    noise_median,
    "{:.3f}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§6.5", "the PTR's quantile offset, 1.645 x that (ppb)", 0.053, offset, "{:.3f}", at_rule, when
)
documented(
    "§6.5",
    "PTR 60 min baseline negative, share of its readings",
    0.65,
    negative,
    "{:.0%}",
    at_rule,
    when,
)
documented(
    "§6.5",
    "the canister's lowest fill (ppb)",
    0.040,
    lowest_fill,
    "{:.3f}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§6.5",
    "fills with the PTR over the same air",
    32,
    int(both.sum()),
    "{}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§6.5",
    "canister minus PTR over the same fill, median (ppb)",
    0.0,
    difference,
    "{:+.3f}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§6.5",
    "canister over PTR over the same fill, median ratio",
    1.01,
    ratio,
    "{:.2f}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§6.5",
    "least-squares slope of canister on PTR",
    0.73,
    slope,
    "{:.2f}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)

# Figure only from here: the whole drive above, then the same span at the baseline's scale.
# A fill is 15 s wide on a five-hour axis, so each is drawn as one marker at its midpoint.
clock = ptr["time"].values
fill_mid = iwas["time"].values
fill_values = iwas["benzene_iwas"].values
adopted = base.values[:, -1, 0]
donor_baseline = donor["baseline_benzene_ptr"].values[:, -1, 0]
fig, (ax, axz) = plt.subplots(
    2, 1, figsize=(10, 6.8), sharex=True, gridspec_kw={"height_ratios": [1.3, 2.0], "hspace": 0.35}
)
t, x = break_gaps(clock, ptr["benzene_ptr"].values)
ax.plot(t, x, color=MUTED, lw=0.5, label="PTR benzene")
ax.plot(fill_mid, fill_values, "_", color=C2, ms=9, mew=2, label="canister fill")
ax.set_ylim(0, np.nanpercentile(ptr["benzene_ptr"].values, 99.5) * 1.3)
finish(
    ax,
    f"{DRIVE_DAY}: the canister's fills, and the PTR's benzene baseline they adopt",
    "one benzene field, two instruments; above, the PTR clipped at its 99.5th percentile; "
    "below, the baseline's scale, zero marked",
    "benzene (ppb)",
    None,
    legend=True,
    loc="upper left",
)
axz.plot(t, x, color=MUTED, lw=0.4, alpha=0.25, label="PTR benzene (faint)")
t, b = break_gaps(clock, donor_baseline)
axz.plot(t, b, color=C1, lw=1.6, label=f"PTR baseline at {ADOPT_WINDOWS[-1]}")
axz.plot(fill_mid, fill_values, "_", color=C2, ms=9, mew=2, label="canister fill")
axz.plot(
    fill_mid,
    adopted,
    "v",
    color=C3,
    ms=6,
    label=f"adopted baseline at each fill ({ADOPT_WINDOWS[-1]})",
)
top_of_zoom = 1.3 * max(float(np.nanmax(donor_baseline)), float(np.nanpercentile(fill_values, 75)))
bottom_of_zoom = min(0.0, float(np.nanmin(donor_baseline)), float(np.nanmin(adopted)))
axz.set_ylim(bottom_of_zoom - 0.08 * (top_of_zoom - bottom_of_zoom), top_of_zoom)
axz.axhline(0, color=INK2, lw=0.8)
above = int(np.sum(fill_values > top_of_zoom))
axz.text(
    0.99,
    0.96,
    f"{above} of {len(fill_values)} fills lie above this panel",
    transform=axz.transAxes,
    ha="right",
    va="top",
    fontsize=8.5,
    color=INK2,
    bbox={"facecolor": SURFACE, "edgecolor": "none", "pad": 2},
)
finish(axz, None, None, "benzene (ppb)")
axz.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=4)
hhmm(axz)
plt.show()

**What to read off the output.** The table is the argument for options: a
canister fill sits alone in a 2 min window nine times out of ten and in a
10 min window more than half the time, and only a window of hours holds the
twenty fills a 5th percentile wants. On the drive, the canister's own quantile
is blank at every fill at 10 min and 60 min, and the adopted baseline gives
each fill the PTR's benzene background over its own window instead, with the
join's record beside it. The lower panel is where to look: at the baseline's
scale the fills sit a few tenths of a ppb above the PTR's 60 min background,
and each adopted value is that background at the fill. That background dips
below zero for much of the drive: a 5th percentile of noisy readings sits below
the true background, and the PTR's benzene background is close to its noise.

The numbers printed above are what the adoption hands the canister, which
notebook 05 §5 showed on generated air: the PTR's quantile offset, about
0.05 ppb, is in every canister enhancement, and the canister's cleanest fill is
only 0.04 ppb. Detection has to correct it with the PTR's noise, not the
canister's (`METHODS.md` §6.5, §6.8). The assumption `from_field` does rest on
holds on average here: over the same air the two instruments agree to a median
of zero, with the disagreement inside plumes.

**Try it.** `ADOPT_WINDOWS = ("60min", "6h")`: at six hours the canister's own
quantile is reported on part of the drive, and the adopted one is still there
beside it for comparison. `QUANTILE = 0.5`: the median needs two fills, and the
canister's own baseline appears at 60 min.

---
## 4. What a noise scale estimated from the record follows on a drive

**The question.** Until 2026-09-29 the rolling state carried a noise scale for
every reading: the declared or reported sigma when a variable has one, else a
robust estimate from the record itself (`diff_mad`, the median absolute
difference between consecutive readings). It moved to Phase 6, beside
detection, and `METHODS.md` §2.5 gives the measurement that argued for the
move: on a drive, the estimate follows the air. Is that true of the files?

This phase no longer ships the estimator, so its arithmetic is written out in
the toolkit (`diff_mad_estimate`), exactly as the Phase-5 code computed it.

In [ ]:
# ---- PARAMETERS (section 4) --------------------------------------------------
NOISE_WINDOW = "10min"  # the estimator's window; METHODS measured 10 min

In [ ]:
documented_ratios = {"co2": (6.5, 30), "noy": (3.9, 17), "ch4": (2.0, 4.3), "benzene": (1.7, 3.7)}
gases = (
    ("co2", "Picarro CO₂", finite_only(drive["picarro"], "co2"), "co2", "ppm"),
    ("noy", "NOy-LIF", lif, "noy", "ppb"),
    ("ch4", "Picarro CH₄", picarro, "ch4", "ppb"),
    ("benzene", "PTR-MS benzene", ptr, "benzene_ptr", "ppb"),
)
applies = AT_DOCUMENTED_DAY and NOISE_WINDOW == "10min"
print(
    f"{'':16} {'quietest 5 %':>13} {'typical':>9} {'busiest 5 %':>12}   "
    "typical / quietest   busiest / quietest"
)
estimates, raised_any = {}, 0.0
for key, label, stream, variable, units in gases:
    estimate, raised = diff_mad_estimate(stream, variable, NOISE_WINDOW)
    estimates[key] = estimate
    raised_any = max(raised_any, raised)
    values = estimate[np.isfinite(estimate)]
    quiet, typical, busy = np.percentile(values, 5), np.median(values), np.percentile(values, 95)
    print(
        f"{label:16} {quiet:>10.3g} {units:3} {typical:>9.3g} {busy:>12.3g}   "
        f"{typical / quiet:>18.1f}   {busy / quiet:>18.1f}"
    )
    documented(
        "§2.5",
        f"{label}: typical window / quietest 5 %",
        documented_ratios[key][0],
        typical / quiet,
        "{:.1f}",
        applies,
        "on 20240718, 10 min",
    )
    documented(
        "§2.5",
        f"{label}: busiest 5 % / quietest 5 %",
        documented_ratios[key][1],
        busy / quiet,
        "{:.0f}" if key in ("co2", "noy") else "{:.1f}",
        applies,
        "on 20240718, 10 min",
    )
documented(
    "§2.5",
    "readings the quantization floor raised, any of the four",
    0.0,
    raised_any,
    "{:.0%}",
    applies,
    "on 20240718, 10 min",
)

# Figure only from here: the record and its noise estimate through the drive.
clock = picarro["time"].values
values = picarro["ch4"].values
fig, (ax, axn) = plt.subplots(
    2, 1, figsize=(10, 6.2), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.2], "hspace": 0.35}
)
t, x = break_gaps(clock, values)
ax.plot(t, x, color=MUTED, lw=0.6)
finish(
    ax,
    f"{DRIVE_DAY}: the Picarro's methane, and a noise scale estimated from it",
    "the estimate is a rolling median of absolute differences; it rises wherever plumes "
    "crowd its window",
    "CH₄ (ppb)",
)
t, sigma = break_gaps(clock, estimates["ch4"])
axn.plot(t, sigma, color=C1, lw=1.2, label=f"diff_mad over {NOISE_WINDOW}")
finish(axn, None, None, "σ (ppb)", None, legend=True, loc="upper left")
hhmm(axn)
plt.show()

**What to read off the output.** An analyzer's precision does not change
several-fold between two stretches of road; the air does. On this drive the
estimate's typical window reads 1.7 to 6.5 times its quietest ones, and its
busiest up to thirty times, most for CO₂. The lower panel shows it for
methane: the estimate rises wherever plumes crowd the window. On a moving
platform the empirical noise scale is
mostly the atmosphere's variability at the sampling interval, so thresholds
quoted in it would rise and fall with the traffic. That is why it is not in
the rolling state: what detection should mean by noise on a moving platform,
the instrument's precision or the local variability, is Phase 6's first
question (`METHODS.md` §2.5, §6.8). Declaring each analyzer's precision in its
manifest puts a declared figure above the estimate. The quantization floor
never acted: every record here is written in steps far finer than its noise.

**Try it.** `NOISE_WINDOW = "2min"`: a jumpier estimate, and the ratios grow,
since a shorter window is filled by a single plume more often.

---
## 5. Ledger and checks

Every number `docs/METHODS.md` §6 and §2.5 quotes about the archive,
re-measured above, and every ✔ check as it ran.

In [ ]:
ledger = pd.DataFrame(
    [(claim, *row) for claim, row in LEDGER.items()],
    columns=["claim", "METHODS", "documented", "measured", "status"],
)
with pd.option_context("display.max_rows", None, "display.max_colwidth", 70, "display.width", 200):
    print(ledger[["METHODS", "claim", "documented", "measured", "status"]].to_string(index=False))
counts = ledger["status"].value_counts()
print(
    f"\nledger: {counts.get('agrees', 0)} agree, {counts.get('DIFFERS', 0)} differ, "
    f"{counts.get('not compared', 0)} not compared, of {len(ledger)}"
)
print(f"checks: {sum(CHECKS.values())} of {len(CHECKS)} hold")
for claim, held in CHECKS.items():
    if not held:
        print(f"  ✘ {claim}")